In [3]:
from pathlib import Path
import duckdb
import pandas as pd

# =========================================================
# PATHS
# =========================================================
# =========================================================
# PROJECT ROOT
# =========================================================

import os
from pathlib import Path


def resolve_project_root():

    # Used automatically by run_pipeline.py
    env_root = os.environ.get(
        "DATA_ENGINEERING_R62_ROOT"
    )

    if env_root:
        root = Path(env_root).resolve()

        if (
            (root / "data").exists()
            and
            (root / "notebooks").exists()
        ):
            return root


    # Interactive notebook execution
    cwd = Path.cwd().resolve()

    candidates = [
        cwd,
        cwd.parent
    ]

    for candidate in candidates:

        if (
            (candidate / "data").exists()
            and
            (candidate / "notebooks").exists()
        ):
            return candidate


    raise RuntimeError(
        "Could not locate DATA_ENGINEERING_R62 project root."
    )


PROJECT_ROOT = resolve_project_root()

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "\n✅ SUCCESS — Project root resolved."
)

PROCESSED = PROJECT_ROOT / "data" / "processed"
FEATURES = PROJECT_ROOT / "data" / "features"

USER_DAY = (
    PROCESSED /
    "user_day_r62_v1.parquet"
)

GT_USERS = (
    PROCESSED /
    "ground_truth_users_r62.parquet"
)

OUTPUT = (
    FEATURES /
    "malicious_pre_attack_baseline_r62.parquet"
)

user_day_path = str(USER_DAY).replace("\\", "/")
gt_users_path = str(GT_USERS).replace("\\", "/")
output_path = str(OUTPUT).replace("\\", "/")

assert USER_DAY.exists()
assert GT_USERS.exists()

print("User-Day    :", USER_DAY)
print("Ground Truth:", GT_USERS)
print("Output      :", OUTPUT)

print(
    "\n✅ SUCCESS — Malicious pre-attack baseline setup ready."
)

Project root: C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62

✅ SUCCESS — Project root resolved.
User-Day    : C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\user_day_r62_v1.parquet
Ground Truth: C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\ground_truth_users_r62.parquet
Output      : C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\features\malicious_pre_attack_baseline_r62.parquet

✅ SUCCESS — Malicious pre-attack baseline setup ready.


In [5]:
# =========================================================
# BUILD MALICIOUS PRE-ATTACK BASELINES
# =========================================================

con = duckdb.connect()

con.execute(f"""
COPY (

WITH malicious_users AS (

    SELECT
        user,
        scenario,
        start AS attack_start,

        CAST(
            start AS DATE
        ) AS attack_start_day

    FROM read_parquet(
        '{gt_users_path}'
    )
),


-- =========================================================
-- ONLY CLEAN DAYS BEFORE ATTACK START DAY
-- =========================================================

pre_attack_days AS (

    SELECT
        d.*,
        m.scenario,
        m.attack_start,
        m.attack_start_day

    FROM read_parquet(
        '{user_day_path}'
    ) d

    INNER JOIN malicious_users m
        ON d.user = m.user

    WHERE d.day < m.attack_start_day

      AND d.is_weekend = 0

      AND d.logon_count > 0
),


-- =========================================================
-- CORE BASELINE
-- =========================================================

core AS (

    SELECT
        user,

        MIN(scenario)
            AS scenario,

        MIN(attack_start)
            AS attack_start,

        COUNT(*)
            AS baseline_workdays,

        MIN(day)
            AS baseline_first_day,

        MAX(day)
            AS baseline_last_day

    FROM pre_attack_days

    GROUP BY user
),


-- =========================================================
-- EMAIL
-- =========================================================

email_stats AS (

    SELECT
        user,

        COUNT(*)
            AS email_active_days,

        AVG(external_email_count)
            AS email_count_mean_active,

        QUANTILE_CONT(
            external_email_count,
            0.50
        ) AS email_count_p50,

        QUANTILE_CONT(
            external_email_count,
            0.75
        ) AS email_count_p75,

        QUANTILE_CONT(
            external_email_count,
            0.90
        ) AS email_count_p90,

        QUANTILE_CONT(
            external_email_count,
            0.95
        ) AS email_count_p95,

        AVG(external_email_bytes)
            AS email_bytes_mean_active,

        QUANTILE_CONT(
            external_email_bytes,
            0.50
        ) AS email_bytes_p50,

        QUANTILE_CONT(
            external_email_bytes,
            0.75
        ) AS email_bytes_p75,

        QUANTILE_CONT(
            external_email_bytes,
            0.90
        ) AS email_bytes_p90,

        QUANTILE_CONT(
            external_email_bytes,
            0.95
        ) AS email_bytes_p95

    FROM pre_attack_days

    WHERE external_email_count > 0

    GROUP BY user
),


email_mad AS (

    SELECT
        p.user,

        MEDIAN(
            ABS(
                p.external_email_count
                - e.email_count_p50
            )
        ) AS email_count_mad,

        MEDIAN(
            ABS(
                p.external_email_bytes
                - e.email_bytes_p50
            )
        ) AS email_bytes_mad

    FROM pre_attack_days p

    INNER JOIN email_stats e
        ON p.user = e.user

    WHERE p.external_email_count > 0

    GROUP BY p.user
),


-- =========================================================
-- USB
-- =========================================================

usb_stats AS (

    SELECT
        user,

        COUNT(*)
            AS usb_active_days,

        AVG(usb_copy_count)
            AS usb_copy_mean_active,

        QUANTILE_CONT(
            usb_copy_count,
            0.50
        ) AS usb_copy_p50,

        QUANTILE_CONT(
            usb_copy_count,
            0.75
        ) AS usb_copy_p75,

        QUANTILE_CONT(
            usb_copy_count,
            0.90
        ) AS usb_copy_p90,

        QUANTILE_CONT(
            usb_copy_count,
            0.95
        ) AS usb_copy_p95

    FROM pre_attack_days

    WHERE usb_copy_count > 0

    GROUP BY user
),


usb_mad AS (

    SELECT
        p.user,

        MEDIAN(
            ABS(
                p.usb_copy_count
                - u.usb_copy_p50
            )
        ) AS usb_copy_mad

    FROM pre_attack_days p

    INNER JOIN usb_stats u
        ON p.user = u.user

    WHERE p.usb_copy_count > 0

    GROUP BY p.user
),


-- =========================================================
-- HTTP UPLOAD
-- =========================================================

http_stats AS (

    SELECT
        user,

        COUNT(*)
            AS http_upload_active_days,

        AVG(www_upload_count)
            AS http_upload_mean_active,

        QUANTILE_CONT(
            www_upload_count,
            0.50
        ) AS http_upload_p50,

        QUANTILE_CONT(
            www_upload_count,
            0.75
        ) AS http_upload_p75,

        QUANTILE_CONT(
            www_upload_count,
            0.90
        ) AS http_upload_p90,

        QUANTILE_CONT(
            www_upload_count,
            0.95
        ) AS http_upload_p95

    FROM pre_attack_days

    WHERE www_upload_count > 0

    GROUP BY user
),


http_mad AS (

    SELECT
        p.user,

        MEDIAN(
            ABS(
                p.www_upload_count
                - h.http_upload_p50
            )
        ) AS http_upload_mad

    FROM pre_attack_days p

    INNER JOIN http_stats h
        ON p.user = h.user

    WHERE p.www_upload_count > 0

    GROUP BY p.user
)


-- =========================================================
-- FINAL PRE-ATTACK BASELINE
-- =========================================================

SELECT

    m.user,
    m.scenario,
    m.attack_start,

    c.baseline_first_day,
    c.baseline_last_day,

    COALESCE(
        c.baseline_workdays,
        0
    ) AS baseline_workdays,


    -- =====================================================
    -- EMAIL
    -- =====================================================

    COALESCE(
        e.email_active_days,
        0
    ) AS email_active_days,

    CASE
        WHEN c.baseline_workdays > 0
        THEN
            COALESCE(
                e.email_active_days,
                0
            )::DOUBLE
            / c.baseline_workdays
        ELSE 0
    END AS email_active_rate,

    e.email_count_mean_active,
    e.email_count_p50,
    e.email_count_p75,
    e.email_count_p90,
    e.email_count_p95,
    em.email_count_mad,

    e.email_bytes_mean_active,
    e.email_bytes_p50,
    e.email_bytes_p75,
    e.email_bytes_p90,
    e.email_bytes_p95,
    em.email_bytes_mad,


    -- =====================================================
    -- USB
    -- =====================================================

    COALESCE(
        u.usb_active_days,
        0
    ) AS usb_active_days,

    CASE
        WHEN c.baseline_workdays > 0
        THEN
            COALESCE(
                u.usb_active_days,
                0
            )::DOUBLE
            / c.baseline_workdays
        ELSE 0
    END AS usb_active_rate,

    u.usb_copy_mean_active,
    u.usb_copy_p50,
    u.usb_copy_p75,
    u.usb_copy_p90,
    u.usb_copy_p95,
    um.usb_copy_mad,


    -- =====================================================
    -- HTTP
    -- =====================================================

    COALESCE(
        h.http_upload_active_days,
        0
    ) AS http_upload_active_days,

    CASE
        WHEN c.baseline_workdays > 0
        THEN
            COALESCE(
                h.http_upload_active_days,
                0
            )::DOUBLE
            / c.baseline_workdays
        ELSE 0
    END AS http_upload_active_rate,

    h.http_upload_mean_active,
    h.http_upload_p50,
    h.http_upload_p75,
    h.http_upload_p90,
    h.http_upload_p95,
    hm.http_upload_mad


FROM malicious_users m

LEFT JOIN core c
    ON m.user = c.user

LEFT JOIN email_stats e
    ON m.user = e.user

LEFT JOIN email_mad em
    ON m.user = em.user

LEFT JOIN usb_stats u
    ON m.user = u.user

LEFT JOIN usb_mad um
    ON m.user = um.user

LEFT JOIN http_stats h
    ON m.user = h.user

LEFT JOIN http_mad hm
    ON m.user = hm.user

ORDER BY m.scenario

)

TO '{output_path}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
);
""")

con.close()

assert OUTPUT.exists()

print("Created:")
print(OUTPUT)

print(
    "\n✅ SUCCESS — Malicious pre-attack baselines created."
)

Created:
C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\features\malicious_pre_attack_baseline_r62.parquet

✅ SUCCESS — Malicious pre-attack baselines created.


In [7]:
# =========================================================
# INSPECT MALICIOUS PRE-ATTACK BASELINES
# =========================================================

con = duckdb.connect()

summary = con.execute(f"""
SELECT

    scenario,
    user,

    attack_start,

    baseline_first_day,
    baseline_last_day,
    baseline_workdays,

    email_active_days,
    usb_active_days,
    http_upload_active_days

FROM read_parquet(
    '{output_path}'
)

ORDER BY scenario
""").df()

con.close()

print("=" * 110)
print("MALICIOUS PRE-ATTACK BASELINES")
print("=" * 110)

print(
    summary.to_string(
        index=False
    )
)

print(
    "\n✅ SUCCESS — Pre-attack baseline summary inspected."
)

MALICIOUS PRE-ATTACK BASELINES
 scenario    user        attack_start baseline_first_day baseline_last_day  baseline_workdays  email_active_days  usb_active_days  http_upload_active_days
        1 ACM2278 2010-08-18 21:47:42         2010-01-04        2010-08-17                159                104                1                       35
        2 CMP2946 2011-02-07 12:28:06         2010-01-04        2011-02-04                276                196               22                       87
        3 PLJ1771 2010-08-12 08:09:00         2010-01-04        2010-08-11                155                 97                0                       28
        4 CDE1846 2011-02-21 11:43:39         2010-01-04        2011-02-18                286                169                0                        0
        5 MBG3183 2010-10-12 13:21:59         2010-01-04        2010-10-11                197                103               43                       44

✅ SUCCESS — Pre-attack baseline summar

In [9]:
# =========================================================
# PRE-ATTACK TEMPORAL LEAKAGE VALIDATION
# =========================================================

con = duckdb.connect()

summary = con.execute(f"""
SELECT

    COUNT(*) AS rows,

    COUNT(
        DISTINCT user
    ) AS users,

    COUNT(
        DISTINCT scenario
    ) AS scenarios,

    MIN(baseline_workdays)
        AS min_baseline_workdays,

    MEDIAN(baseline_workdays)
        AS median_baseline_workdays,

    MAX(baseline_workdays)
        AS max_baseline_workdays

FROM read_parquet(
    '{output_path}'
)
""").df()


leakage = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{output_path}'
)

WHERE baseline_last_day
      >= CAST(
          attack_start AS DATE
      )
""").fetchone()[0]


duplicate_users = con.execute(f"""
SELECT COUNT(*)

FROM (

    SELECT
        user,
        COUNT(*) AS n

    FROM read_parquet(
        '{output_path}'
    )

    GROUP BY user

    HAVING COUNT(*) > 1
)
""").fetchone()[0]


bad_rates = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{output_path}'
)

WHERE email_active_rate NOT BETWEEN 0 AND 1

   OR usb_active_rate NOT BETWEEN 0 AND 1

   OR http_upload_active_rate NOT BETWEEN 0 AND 1
""").fetchone()[0]


con.close()


print("=" * 100)
print("MALICIOUS PRE-ATTACK BASELINE VALIDATION")
print("=" * 100)

print(
    summary.to_string(
        index=False
    )
)

print("\nTemporal leakage :", leakage)
print("Duplicate users  :", duplicate_users)
print("Invalid rates    :", bad_rates)


assert int(summary.loc[0, "rows"]) == 5
assert int(summary.loc[0, "users"]) == 5
assert int(summary.loc[0, "scenarios"]) == 5

assert leakage == 0
assert duplicate_users == 0
assert bad_rates == 0


print(
    "\n✅ SUCCESS — MALICIOUS PRE-ATTACK BASELINES PASSED VALIDATION."
)

MALICIOUS PRE-ATTACK BASELINE VALIDATION
 rows  users  scenarios  min_baseline_workdays  median_baseline_workdays  max_baseline_workdays
    5      5          5                    155                     197.0                    286

Temporal leakage : 0
Duplicate users  : 0
Invalid rates    : 0

✅ SUCCESS — MALICIOUS PRE-ATTACK BASELINES PASSED VALIDATION.
